In [28]:
## Load libraries
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from keras.datasets import  mnist
plt.style.use('dark_background')
%matplotlib inline

In [29]:
np.set_printoptions(precision=2)
import tensorflow as tf
tf.__version__

'2.14.0'

In [30]:
#Load MNIST data
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.transpose(1, 2, 0)
X_test = X_test.transpose(1, 2, 0)
X_train = X_train.reshape(X_train.shape[0] * X_train.shape[1], X_train.shape[2])
X_test = X_test.reshape(X_test.shape[0] * X_test.shape[1], X_test.shape[2])


In [31]:
num_labels = len(np.unique(y_train))
num_features = X_train.shape[0]
num_samples = X_train.shape[1]

In [32]:
#One hot encode class labels
Y_train = tf.keras.utils.to_categorical(y_train).T
Y_test = tf.keras.utils.to_categorical(y_test).T


In [33]:
#Normalizze the samples (images)
xmax = np.amax(X_train)
xmin = np.amin(X_train)
X_train = (X_train - xmin)/(xmax - xmin) #all train features turn into a number between 1 and 0
X_test = (X_test - xmin)/(xmax - xmin) 

In [34]:
print('MNIST set')
print('----------')
print('Number of training samples = %d'%(num_samples))
print('Number of features = %d'%(num_features))
print('Number of output classes = %d'%(num_labels))

MNIST set
----------
Number of training samples = 60000
Number of features = 784
Number of output classes = 10


In [35]:
class Layer:
  def __init__(self):
    self.input = None
    self.output = None

  def forward(self, input):
    pass

  def backward(self, output_gradient, learning_rate):
    pass

In [36]:
## Define the loss function and its gradient
def cce(Y, Yhat):
  return(np.mean(np.sum(-Y*np.log(Yhat), axis=0)))

def cce_gradient(Y, Yhat):
  return(-Y/Yhat)

# TensorFlow in-built function for categorical crossentropy loss
#cce = tf.keras.losses.CategoricalCrossentropy()

In [37]:
class Softmax(Layer):
    def forward(self, input):
        self.input = input  # Assign the input to self.input
        # Ensure that input is a NumPy array before applying softmax
        input_np = np.array(input)
        self.output = tf.nn.softmax(input_np, axis=0).numpy()

    def backward(self, output_gradient, learning_rate=None):
        # Following is the inefficient way of calculating the backward gradient
        softmax_gradient = np.empty((self.input.shape[0], output_gradient.shape[1]), dtype=np.float64)
        for b in range(softmax_gradient.shape[1]):
            softmax_gradient[:, b] = np.dot((np.identity(self.output.shape[0]) - self.output[:, b].T) * self.output[:, b],
                                            output_gradient[:, b])
        return softmax_gradient


In [45]:
## Dense layer class
class Dense(Layer):
    def __init__(self, input_size, output_size):
        self.weights = np.empty((output_size, input_size + 1), dtype=np.float64) # +1 for bias trick
        self.weights[:, :-1]  = 0.01 * np.random.randn(output_size, input_size) # 
        self.weights[:, -1] = 0.01 # Set all bias values to the same nonzero constant

    def forward(self, input):
        self.input = np.vstack([input, np.ones((1, input.shape[1]))]) #bias trick
        self.output = np.dot(self.weights, self.input)

    def backward(self, output_gradient, learning_rate=None):
        dense_gradient = np.zeros((self.output.shape[0], self.input.shape[0]), dtype=np.float64)
        for b in range(output_gradient.shape[1]):
            dense_gradient += np.dot(output_gradient[:, b].reshape(-1, 1), self.input[:, b].reshape(1, -1))
        dense_gradient = np.mean(dense_gradient, axis=1)
        dense_gradient = (1 / output_gradient.shape[1]) * dense_gradient
        # self.weights -= learning_rate * dense_gradient
        self.weights[:, :-1] -= learning_rate * dense_gradient[:, :-1]
        self.weights[:, -1] -= learning_rate * dense_gradient[:, -1]

        return np.dot(self.weights.T, output_gradient)

        ## Following is the efficient way of calculating the backward gradient
        #dense_gradient = (1/output_gradient.shape[1])1])*np.dot(np.atleast_2d(output_gradient), np.atleast_2d(self.input).T)
        #self.weights = learning_rate * (-dense_gradient)

In [47]:
#Function to generate sample indices for batch processing according to batch size
def generate_batch_indices(num_samples, batch_size):
    #reorder sample indices
    reordered_sample_indices= np.random.choice(num_samples, num_samples, replace=False)
    #Generate batch indices for batch processing
    batch_indices = np.split(reordered_sample_indices, np.arange(batch_size, len(reordered_sample_indices), batch_size))
    return batch_indices

In [48]:
#Example generation of batch indices
num_samples = 64
batch_size = 8
batch_indices = generate_batch_indices(num_samples, batch_size)
print(batch_indices)

[array([48, 33, 42,  4, 29, 15, 45, 36]), array([19, 14, 60, 23, 32, 56, 40, 50]), array([27,  3, 57, 10, 31,  5, 26, 52]), array([58, 30, 12, 53,  7, 59, 16, 17]), array([55, 38, 51, 22, 44, 46, 20, 43]), array([35, 49, 62, 24,  0,  1, 18, 34]), array([54,  2, 47, 25, 11, 39,  9, 41]), array([ 6, 61, 13, 63, 28, 37,  8, 21])]


In [49]:
#Train the 0-layer neural network using batch training with batch size = 16
learning_rate = 1e-2
batch_size = 200
nepochs = 20
loss_epoch = np.empty(nepochs,dtype=np.float64) #create empty array to store loss values for each epoch


In [50]:
Z = np.arange(6).reshape(2, -1).astype(np.float64)
print(Z)
tf.nn.softmax(Z).numpy()

[[0. 1. 2.]
 [3. 4. 5.]]


array([[0.09, 0.24, 0.67],
       [0.09, 0.24, 0.67]])

In [51]:
#Neural network architecture
dlayer = Dense(num_features, num_labels)
softmax = Softmax()

In [52]:
epoch = 0
while epoch < nepochs:
    batch_indices = generate_batch_indices(num_samples, batch_size)
    loss = 0
    for b in range(len(batch_indices)):
        dlayer.forward(X_train[:, batch_indices[b]])
        softmax.forward(dlayer.output)
        loss += cce(Y_train[:, batch_indices[b]], softmax.output)

        # backward prop starts here
        grad = cce_gradient(Y_train[:, batch_indices[b]], softmax.output)
        grad = softmax.backward(grad)
        grad = dlayer.backward(grad, learning_rate)

    loss_epoch[epoch] = loss/len(batch_indices)
    print('Epoch %d: loss = %f'%(epoch+1, loss_epoch[epoch]))
    epoch = epoch + 1

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
##Plot training loss as a function of epoch
plt.plot(loss_epoch)
plt.xlabel('Epoch')
plt.ylabel('Loss value')
plt.show()

In [ ]:
#Accuracy on test set
dlayer.forward(X_test)
softmax.forward(dlayer.output)
ypred = np.argmax(softmax.output.T, axis=1)
print(ypred)
ytrue = np.argmax(Y_test.T, axis=1)
print(ytrue)
np.mean(ytrue==ypred)

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
sample0 = X_train[0, :, :]
print(X_train.shape)
X_train_reshape = X_train.reshape(X_train.shape[1] * X_train.shape[2], X_train.shape[0])
X_train.reshape[:, 0].reshape(28, 28)